In [1]:
import io
import pandas as pd

# Dataset 1
csv_data_1 = """Product,January Units Sold,February Units Sold,March Units Sold
Printable Wall Art,25,32,41
Digital Planners,40,28,35
Budget Trackers,18,24,30
Coloring Pages,12,19,27"""

with open("dataset1_digital_products.csv", "w") as f:
  f.write(csv_data_1)

# Dataset 2
csv_data_2 = """Product,January,February,March,April,May
Laptop Stand,42,51,47,63,58
Wireless Mouse,75,81,69,92,88
Keyboard,54,61,57,73,70"""

with open("dataset2_electronics.csv", "w") as f:
  f.write(csv_data_2)

# Dataset 3
csv_data_3 = """Borough,Year,Population,White,% W,Non-Hispanic Whites,% ANG,Black,% B,Asian,% A,Other or Mixed,% O/M,Hispanic/Latino,% H/L
Brooklyn,1920,"2,018,356","1,984,953",98.35,N/A,N/A,"31,912",1.58,"1,459",0.07,32,0,N/A,N/A
Brooklyn,1970,"2,602,012","1,905,788",73.24,N/A,N/A,"656,194",25.22,"16,138",0.62,"23,892",0.92,N/A,N/A
Brooklyn,2010,"2,504,700","1,072,041",42.80,"893,306",35.67,"860,083",34.34,"263,519",10.52,"309,057",12.34,"496,285",19.81
Queens,2010,"2,230,722","888,418",39.82,"616,720",27.65,"426,682",19.13,"511,596",22.93,"192,209",8.62,"613,750",27.51"""

with open("dataset3_nyc_demographics.csv", "w") as f:
  f.write(csv_data_3)

print("All CSV files created successfully!")

All CSV files created successfully!


# Dataset 1: Digital Products Monthly Sales

### 1. Load Wide Data
We begin by reading the un-tidied CSV into a pandas DataFrame.

In [2]:
import pandas as pd

# Load raw CSV
df1_raw = pd.read_csv('dataset1_digital_products.csv')
print("--- Raw Dataset 1 (Wide Format) ---")
display(df1_raw)

--- Raw Dataset 1 (Wide Format) ---


,Product,January Units Sold,February Units Sold,March Units Sold
0,Printable Wall Art,25,32,41
1,Digital Planners,40,28,35
2,Budget Trackers,18,24,30
3,Coloring Pages,12,19,27


### 2. Data Tidying & Transformation
This dataset is untidy because month names are spread across column headers rather than stored as values in a single column.
We will use `pd.melt()` to collapse the month columns into row observations, clean the month strings, and ensure integer data types.

In [3]:
# Melt wide columns into long format
df1_tidy = pd.melt(
    df1_raw,
    id_vars=['Product'],
    var_name='Month',
    value_name='Units_Sold',
)

# Clean Month column strings
df1_tidy['Month'] = df1_tidy['Month'].str.replace(' Units Sold', '')

# Ensure correct integer data type
df1_tidy['Units_Sold'] = df1_tidy['Units_Sold'].astype(int)

print("--- Tidied Dataset 1 (Long Format) ---")
display(df1_tidy)

--- Tidied Dataset 1 (Long Format) ---


,Product,Month,Units_Sold
0,Printable Wall Art,January,25
1,Digital Planners,January,40
2,Budget Trackers,January,18
3,Coloring Pages,January,12
4,Printable Wall Art,February,32
5,Digital Planners,February,28
6,Budget Trackers,February,24
7,Coloring Pages,February,19
8,Printable Wall Art,March,41
9,Digital Planners,March,35


### 3. Analysis & Conclusions
We perform the requested analysis to examine total sales per product across the three months and evaluate monthly growth trends.

In [4]:
# Total sales by product
total_sales_df1 = (
    df1_tidy.groupby('Product')['Units_Sold']
    .sum()
    .reset_index()
    .sort_values(by='Units_Sold', ascending=False)
)
print("--- Total Units Sold Per Product ---")
display(total_sales_df1)

# Monthly trends
pivot_df1 = df1_tidy.pivot(
    index='Product', columns='Month', values='Units_Sold'
)[['January', 'February', 'March']]
print("\n--- Monthly Sales Trend ---")
display(pivot_df1)

--- Total Units Sold Per Product ---


,Product,Units_Sold
2,Digital Planners,103
3,Printable Wall Art,98
0,Budget Trackers,72
1,Coloring Pages,58



--- Monthly Sales Trend ---


Month,January,February,March
Product,,,
Budget Trackers,18,24,30
Coloring Pages,12,19,27
Digital Planners,40,28,35
Printable Wall Art,25,32,41


# Dataset 2: Online Store Electronics Sales

### 1. Load Wide Data
Read the raw wide CSV containing monthly sales figures for electronic items.

In [5]:
# Load raw CSV
df2_raw = pd.read_csv('dataset2_electronics.csv')
print("--- Raw Dataset 2 (Wide Format) ---")
display(df2_raw)

--- Raw Dataset 2 (Wide Format) ---


,Product,January,February,March,April,May
0,Laptop Stand,42,51,47,63,58
1,Wireless Mouse,75,81,69,92,88
2,Keyboard,54,61,57,73,70


### 2. Data Tidying & Transformation
We reshape the wide month headers (`January` through `May`) into a single variable column named `Month` using `pd.melt()`.

In [6]:
# Unpivot wide month columns
df2_tidy = pd.melt(
    df2_raw, id_vars=['Product'], var_name='Month', value_name='Units_Sold'
)

print("--- Tidied Dataset 2 (Long Format) ---")
display(df2_tidy)

--- Tidied Dataset 2 (Long Format) ---


,Product,Month,Units_Sold
0,Laptop Stand,January,42
1,Wireless Mouse,January,75
2,Keyboard,January,54
3,Laptop Stand,February,51
4,Wireless Mouse,February,81
5,Keyboard,February,61
6,Laptop Stand,March,47
7,Wireless Mouse,March,69
8,Keyboard,March,57
9,Laptop Stand,April,63


### 3. Analysis & Conclusions
We summarize sales performance for each electronic item (total, average, minimum, and maximum units sold per month) to identify top performers.

In [7]:
# Aggregated summary statistics per product
df2_summary = (
    df2_tidy.groupby('Product')['Units_Sold']
    .agg(
        Total_Units='sum',
        Average_Monthly='mean',
        Lowest_Month='min',
        Highest_Month='max',
    )
    .reset_index()
)

display(df2_summary)

,Product,Total_Units,Average_Monthly,Lowest_Month,Highest_Month
0,Keyboard,315,63.0,54,73
1,Laptop Stand,261,52.2,42,63
2,Wireless Mouse,405,81.0,69,92


# Dataset 3: NYC Borough Demographics

### 1. Load Wide Data
Load the demographic historical dataset for NYC boroughs.

In [8]:
# Load raw CSV
df3_raw = pd.read_csv('dataset3_nyc_demographics.csv')
print("--- Raw Dataset 3 (Wide Format) ---")
display(df3_raw)

--- Raw Dataset 3 (Wide Format) ---


,Borough,Year,Population,White,% W,Non-Hispanic Whites,% ANG,Black,% B,Asian,% A,Other or Mixed,% O/M,Hispanic/Latino,% H/L
0,Brooklyn,1920,"2,018,356","1,984,953",98.35,NaN,NaN,"31,912",1.58,"1,459",0.07,32,0.00,NaN,NaN
1,Brooklyn,1970,"2,602,012","1,905,788",73.24,NaN,NaN,"656,194",25.22,"16,138",0.62,"23,892",0.92,NaN,NaN
2,Brooklyn,2010,"2,504,700","1,072,041",42.80,"893,306",35.67,"860,083",34.34,"263,519",10.52,"309,057",12.34,"496,285",19.81
3,Queens,2010,"2,230,722","888,418",39.82,"616,720",27.65,"426,682",19.13,"511,596",22.93,"192,209",8.62,"613,750",27.51


### 2. Data Tidying & Transformation
This dataset contains string numbers with commas and `N/A` text entries, as well as multiple demographic count columns.
We clean the numeric values, melt demographic group headers into a single categorical variable, calculate structured percentage shares, and remove historical `NaN` missing values.

In [9]:
import numpy as np

# List of numeric count columns to clean
count_cols = [
    'White',
    'Non-Hispanic Whites',
    'Black',
    'Asian',
    'Other or Mixed',
    'Hispanic/Latino',
]

# Clean strings, remove commas, handle 'N/A'
df3_clean = df3_raw.copy()
df3_clean['Population'] = (
    df3_clean['Population'].astype(str).str.replace(',', '').str.strip()
)
df3_clean['Population'] = pd.to_numeric(df3_clean['Population'])

for col in count_cols:
  df3_clean[col] = (
      df3_clean[col].astype(str).str.replace(',', '').str.strip()
  )
  df3_clean[col] = pd.to_numeric(
      df3_clean[col].replace('N/A', np.nan), errors='coerce'
  )

# Melt count columns into tidy format
df3_tidy = pd.melt(
    df3_clean,
    id_vars=['Borough', 'Year', 'Population'],
    value_vars=count_cols,
    var_name='Demographic_Group',
    value_name='Count',
)

# Calculate explicit percentage variable
df3_tidy['Percentage'] = (
    df3_tidy['Count'] / df3_tidy['Population'] * 100
).round(2)

# Drop missing historical measurements
df3_tidy = df3_tidy.dropna(subset=['Count']).reset_index(drop=True)

print("--- Tidied Dataset 3 (Long Format) ---")
display(df3_tidy.head(15))

--- Tidied Dataset 3 (Long Format) ---


,Borough,Year,Population,Demographic_Group,Count,Percentage
0,Brooklyn,1920,2018356,White,1984953.0,98.35
1,Brooklyn,1970,2602012,White,1905788.0,73.24
2,Brooklyn,2010,2504700,White,1072041.0,42.80
3,Queens,2010,2230722,White,888418.0,39.83
4,Brooklyn,2010,2504700,Non-Hispanic Whites,893306.0,35.67
5,Queens,2010,2230722,Non-Hispanic Whites,616720.0,27.65
6,Brooklyn,1920,2018356,Black,31912.0,1.58
7,Brooklyn,1970,2602012,Black,656194.0,25.22
8,Brooklyn,2010,2504700,Black,860083.0,34.34
9,Queens,2010,2230722,Black,426682.0,19.13


### 3. Analysis & Conclusions
We pivot the tidied dataset to analyze how demographic shares shifted across census years in Brooklyn.

In [10]:
# Demographic percentage shifts over time in Brooklyn
bk_shift = df3_tidy[df3_tidy['Borough'] == 'Brooklyn'].pivot(
    index='Demographic_Group', columns='Year', values='Percentage'
)

print("--- Brooklyn Demographic Share (%) Shift Over Time ---")
display(bk_shift)

--- Brooklyn Demographic Share (%) Shift Over Time ---


Year,1920,1970,2010
Demographic_Group,,,
Asian,0.07,0.62,10.52
Black,1.58,25.22,34.34
Hispanic/Latino,NaN,NaN,19.81
Non-Hispanic Whites,NaN,NaN,35.67
Other or Mixed,0.00,0.92,12.34
White,98.35,73.24,42.80
